In [33]:
import tensorflow as tf
print(tf.__version__)

2.17.0


In [34]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [35]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load data
df = pd.read_csv(r'C:\Users\jecroisp\Documents\Classes\AI Classial way\classicalAiProjectEP\lib\data\Cleaned_Tweets_Dataset.csv')  # Adjust path accordingly

# Extract tweets and sentiments
texts = df['text'].astype(str)
labels = df['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)  # Convert sentiments to binary labels

# Tokenize text
tokenizer = Tokenizer(num_words=1000)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

# Pad sequences to ensure uniform input size
max_seq_length = max(len(x) for x in sequences)
X = pad_sequences(sequences, maxlen=max_seq_length)
y = labels.values


TF-IDF Vectorizer For accuracy score purposes 

In [41]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
texts = df['text'].values.astype('U')

In [42]:
tfidf = TfidfVectorizer(strip_accents=None, lowercase=False, preprocessor=None)
X = tfidf.fit_transform(texts)

In [43]:

y = df['value']


In [44]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer


tfidf = TfidfVectorizer(strip_accents=None, lowercase=False, preprocessor=None)
tfidf.fit(df['text'].values.astype('U'))
joblib.dump(tfidf, r'C:\Users\jecroisp\Documents\Classes\AI Classial way\classicalAiProjectEP\lib\models\tfidf_vectorizer.pkl')  # Save the newly fitted vectorizer

# Load the pre-fitted vectorizer
tfidf = joblib.load(r'C:\Users\jecroisp\Documents\Classes\AI Classial way\classicalAiProjectEP\lib\models\tfidf_vectorizer.pkl')

# Transform the data
X = tfidf.transform(df['text'].values.astype('U'))

# Now you can use X for your model training or prediction tasks


WELCOME - PArt 1 is the LSTM Model Part 2 is Random Forrest. Lets get to the first part. 

In [36]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Define the model
model = Sequential()
model.add(Embedding(input_dim=1000, output_dim=32, input_length=max_seq_length))
model.add(LSTM(50))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X, y, epochs=10, batch_size=32)


Epoch 1/10


c:\Users\jecroisp\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


859/859 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.7604 - loss: 0.5108
Epoch 2/10
859/859 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8516 - loss: 0.3716
Epoch 3/10
859/859 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8582 - loss: 0.3553
Epoch 4/10
859/859 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8635 - loss: 0.3470
Epoch 5/10
859/859 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8664 - loss: 0.3393
Epoch 6/10
859/859 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8737 - loss: 0.3245
Epoch 7/10
859/859 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8742 - loss: 0.3163
Epoch 8/10
859/859 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8815 - loss: 0.3064
Epoch 9/10
859/859 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8877 - loss: 0.2895
Epoch 10/10
859/859 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8872 - loss: 0.2830


In [37]:
model.save(r'C:\Users\jecroisp\Documents\Classes\AI Classial way\classicalAiProjectEP\lib\models\sentiment_lstm_model.h5')


 MODEL SPLIT HERE - Next Model is RANDOM FOREST


In [45]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [39]:
pip install --upgrade scikit-learn==1.5.1


Note: you may need to restart the kernel to use updated packages.


In [40]:
from sklearn.ensemble import RandomForestClassifier
import joblib
from sklearn.model_selection import train_test_split
import pandas as pd


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
rf_model.fit(X_train, y_train)

# Save the model
joblib.dump(rf_model, r'C:\Users\jecroisp\Documents\Classes\AI Classial way\classicalAiProjectEP\lib\models\random_forest_model.pkl')


['C:\\Users\\jecroisp\\Documents\\Classes\\AI Classial way\\classicalAiProjectEP\\lib\\models\\random_forest_model.pkl']

In [48]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import joblib


# Prepare TF-IDF for Random Forest
tfidf = TfidfVectorizer(strip_accents=None, lowercase=False, preprocessor=None)
X_tfidf = tfidf.fit_transform(df['text'].values.astype('U'))
y = df['value'].apply(lambda x: 1 if x == 'positive' else 0)  # Adjust this as per your label column
X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(X_tfidf, y, test_size=0.3, random_state=42)

# Prepare Tokenized data for LSTM
tokenizer = Tokenizer(num_words=1000)
tokenizer.fit_on_texts(df['text'].astype(str))
sequences = tokenizer.texts_to_sequences(df['text'].astype(str))
X_seq = pad_sequences(sequences, maxlen=100)  # max length is set to 100, adjust as needed
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(X_seq, y, test_size=0.3, random_state=42)

# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
rf_model.fit(X_train_tfidf, y_train_tfidf)
joblib.dump(rf_model, 'random_forest_model.pkl')

# Define LSTM Model
model = Sequential()
model.add(Embedding(input_dim=1000, output_dim=32, input_length=100))
model.add(LSTM(50))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train_seq, y_train_seq, epochs=10, batch_size=32)
model.save('sentiment_lstm_model.keras')

# Output model training summaries and save model appropriately
print("Random Forest and LSTM models trained and saved.")


Epoch 1/10


c:\Users\jecroisp\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


602/602 ━━━━━━━━━━━━━━━━━━━━ 13s 19ms/step - accuracy: 0.9895 - loss: 0.0625
Epoch 2/10
602/602 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 1.0000 - loss: 1.1646e-04
Epoch 3/10
602/602 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 1.0000 - loss: 4.5142e-05
Epoch 4/10
602/602 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 1.0000 - loss: 2.3601e-05
Epoch 5/10
602/602 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 1.0000 - loss: 1.4246e-05
Epoch 6/10
602/602 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 1.0000 - loss: 9.2510e-06
Epoch 7/10
602/602 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 1.0000 - loss: 6.2724e-06
Epoch 8/10
602/602 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 1.0000 - loss: 4.3705e-06
Epoch 9/10
602/602 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 1.0000 - loss: 3.1021e-06
Epoch 10/10
602/602 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 1.0000 - loss: 2.2301e-06
Random Forest and LSTM models trained and saved.
